# Feature Selection - Ridge Regression and LASSO
In this notebook we will cover two modified regression approaches that select a subset of the X variables to include in the final model. We will estimate regressions, ridge regression and LASSO and then compare the MSE of predictions from those models. We will also talk about splitting our data into training and testing subsets. 

In [ ]:
# Import Packaes

%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import scale, PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV, Lasso, LassoCV
from sklearn.metrics import mean_squared_error

In [ ]:
# Import data
url = "https://raw.githubusercontent.com/franciscadias/data/master/kc_house_data.csv"   # Data from a kaggle competition 
df = pd.read_csv(url)
#df.info()

X = df.drop(["price", "date", "id", "lat", "long"], axis=1).copy()                  # Create a new version of the data with numeric varaibles

for col in list(X):                                                                  # Convert everything to be a float
    X[col] = X[col].astype(float)

# Create dummy variables for categorical variables
X=pd.get_dummies(X, columns=['zipcode'])

# Create interactions and squares for every variable for increased feature representation    
#poly = PolynomialFeatures(2)
#poly_var = poly.fit_transform(X)
#X.join(poly_var, lsuffix='_l', rsuffix='_r')
#X=np.hstack((X,poly_var))
    
print(X.head())

# Define the dependent variable. Notice the log here!
y = np.log(df["price"])
df["log_price"] = y
#print(y.head())

## Overfitting

Overfitting occurs when we use too many variables in our model. This improves the in sample prediction, but makes out of sample prediction worse. The easiest way to prevent this is to split our data set into two parts. We will fit the model on part of the data, then test on different data. If the in-sample prediction is very good, but the out of sample prediction on the test portion of the data set is poor, then we know we have overfit our model. If that is the case we will need to reduce the number of variables. The algorithms here help us to decide which variables are most important and which can be deemphasized. 

We are working with regression-style algorithms, but the concept of splitting the data into train and test sub-samples is used across data science projects. 

In [ ]:
## Use test/train split (from sklearn)  to separate the data

X_train, X_test , y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=50)

# Regression
Start by estimating a traditional linear regression and saving the outcome variables for anlaysis 

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)                         # Fit a standard regression on the training data
pred2 = lr.predict(X_test)                       # Use this model to predict the test data
print(pd.Series(lr.coef_, index = X.columns))    # Print coefficients
print(mean_squared_error(y_test, pred2))         # Calculate the test MSE

# Ridge Regression 


In [ ]:
# Generate different L1 penalities
alphas = 10**np.linspace(10,-2,100)*0.5
#print(alphas)

ridge = Ridge()
coefs = []
for a in alphas:
    ridge.set_params(alpha = a)
    ridge.fit(X, y)
    coefs.append(ridge.coef_)
    
np.shape(coefs)

ax = plt.gca()
ax.plot(alphas, coefs)
ax.set_xscale('log')
plt.axis('tight')
plt.xlabel('alpha')
plt.ylabel('weights')



In [ ]:
ridge2 = Ridge(alpha = 4)
ridge2.fit(X_train, y_train)             # Fit a ridge regression on the training data
pred2 = ridge2.predict(X_test)           # Use this model to predict the test data
print(pd.Series(ridge2.coef_, index = X.columns)) # Print coefficients
print(mean_squared_error(y_test, pred2))          # Calculate the test MSE

In [ ]:
ridge2 = Ridge(alpha = 10**10)
ridge2.fit(X_train, y_train)             # Fit a ridge regression on the training data
pred2 = ridge2.predict(X_test)           # Use this model to predict the test data
print(pd.Series(ridge2.coef_, index = X.columns)) # Print coefficients
print(mean_squared_error(y_test, pred2))          # Calculate the test MSE

In [ ]:
# Run the model with alpha = 0 to reproduce the standard regression results
ridge2 = Ridge(alpha = 0)
ridge2.fit(X_train, y_train)             # Fit a ridge regression on the training data
pred2 = ridge2.predict(X_test)           # Use this model to predict the test data
print(pd.Series(ridge2.coef_, index = X.columns)) # Print coefficients
print(mean_squared_error(y_test, pred2))          # Calculate the test MSE

## Finding the best alpha
We can use cross validation to find the alpha that has the lowest MSE out of sample.

In [ ]:
ridgecv = RidgeCV(alphas = alphas, scoring = 'neg_mean_squared_error')
ridgecv.fit(X_train, y_train)
ridgecv.alpha_

In [ ]:
ridge4 = Ridge(alpha = ridgecv.alpha_)
ridge4.fit(X_train, y_train)
mean_squared_error(y_test, ridge4.predict(X_test))

In [ ]:
ridge4.fit(X, y)
pd.Series(ridge4.coef_, index = X.columns)

# LASSO
Now lets use he same data to estimate LASSO. LASSO in equivalent to ridge regression with a infinite penalty. This shrinks the least important coefficients all the way to zero. 

In [ ]:
alphas = np.linspace(10,-2,100)*0.5
print(alphas)

In [ ]:
# Initalize the parameters
coefs = []
alphas = np.linspace(.1,24)
print(alphas)
lasso = Lasso(max_iter = 10000)

for a in alphas:
    #print(a)
    lasso.set_params(alpha=a)
    lasso.fit(X_train, y_train)
    #print(lasso.coef_)
    coefs.append(lasso.coef_)
    
ax = plt.gca()
ax.plot(alphas, coefs)
#ax.set_xscale('log')
plt.axis('tight')
plt.xlabel('alpha')
plt.ylabel('weights')

In [ ]:
lasso.set_params(alpha=.00001)
lasso.fit(X_train, y_train)
print(mean_squared_error(y_test, lasso.predict(X_test)))


# Some of the coefficients are now reduced to exactly zero.
pd.Series(lasso.coef_, index=X.columns)

In [ ]:
lasso.set_params(alpha=52)
lasso.fit(X_train, y_train)
print(mean_squared_error(y_test, lasso.predict(X_test)))


# Some of the coefficients are now reduced to exactly zero.
pd.Series(lasso.coef_, index=X.columns)

In [ ]:
lassocv = LassoCV(alphas = None, cv = 10, max_iter = 100000)
lassocv.fit(X_train, y_train)

lasso.set_params(alpha=lassocv.alpha_)
lasso.fit(X_train, y_train)
print(mean_squared_error(y_test, lasso.predict(X_test)))


# Some of the coefficients are now reduced to exactly zero.
pd.Series(lasso.coef_, index=X.columns)

# Review
Let's look at all three types of regression on the same axis. The code below fits a standard linear regression, two ridge regressions and LASSO. It prints the test and train scores for each model.

In [ ]:
# Fit the linear model

lr = LinearRegression()
lr.fit(X_train, y_train)

# Fit Ridge with low alpha
rr = Ridge(alpha=0.01) # higher the alpha value, more restriction on the coefficients; 
rr.fit(X_train, y_train)

# Fit ridge with high alpha
rr100 = Ridge(alpha=100) #  comparison with alpha value
rr100.fit(X_train, y_train)

# Fit lasso
lasso.set_params(alpha=.0001)
lasso.fit(X_train, y_train)


# Calculate the scores
train_score=lr.score(X_train, y_train)
test_score=lr.score(X_test, y_test)
Ridge_train_score = rr.score(X_train,y_train)
Ridge_test_score = rr.score(X_test, y_test)
Ridge_train_score100 = rr100.score(X_train,y_train)
Ridge_test_score100 = rr100.score(X_test, y_test)
lasso_train_score=lasso.score(X_train,y_train)
lasso_test_score=lasso.score(X_test,y_test)

# Print the scores
print ("linear regression train score:", train_score)
print ("linear regression test score:", test_score)
print ("ridge regression train score low alpha:", Ridge_train_score)
print ("ridge regression test score low alpha:", Ridge_test_score)
print ("ridge regression train score high alpha:", Ridge_train_score100)
print ("ridge regression test score high alpha:", Ridge_test_score100)
print ("lasso train score:", lasso_train_score)
print ("lasso test score:", lasso_test_score)


plt.plot(rr.coef_,alpha=0.7,linestyle='none',marker='*',markersize=5,color='red',label=r'Ridge; $\alpha = 0.01$',zorder=7) # zorder for ordering the markers
plt.plot(rr100.coef_,alpha=0.5,linestyle='none',marker='d',markersize=6,color='blue',label=r'Ridge; $\alpha = 100$') # alpha here is for transparency
#plt.plot(lr.coef_,alpha=0.4,linestyle='none',marker='o',markersize=7,color='green',label='Linear Regression')
plt.plot(lasso.coef_,alpha=0.3,linestyle='none',marker='x',markersize=7,color='cyan',label='LASSO')

plt.xlabel('Coefficient Index',fontsize=16)
plt.ylabel('Coefficient Magnitude',fontsize=16)
plt.legend(fontsize=13,loc=4)
plt.show()